# ViFinQA active-learning open-source GPU cycle V1

Mục tiêu: chạy **Qwen3-8B proposer** và **Mistral-Nemo-12B blind critic** trên cùng 64 packet numeric-free, rồi kiểm tra schema/hash và tạo hàng đợi escalation. Notebook này không dùng ChatGPT, không fine-tune từ output máy, không tạo certificate và không mở release.

Kaggle cần bật **GPU** và **Internet**, đồng thời attach private dataset chứa `active_learning_kaggle_bundle_v3_20260824`. Thành công kỹ thuật là có hai validated JSONL, reconciliation receipt và archive tải về; thành công kỹ thuật không đồng nghĩa đáp án đã được chứng minh.

In [ ]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/Dle28/nlp-finance-query-.git"
SOURCE_COMMIT = "362b0ed6d838df215deeb92a052993c52aae7347"
INPUT_ROOT = Path("/kaggle/input")
RUN_ID = SOURCE_COMMIT[:8]
WORK_ROOT = Path(f"/kaggle/working/vifinqa-active-learning-v1-{RUN_ID}")
REPO_DIR = WORK_ROOT / "repo"
OUTPUT_ROOT = WORK_ROOT / "outputs"

if WORK_ROOT.exists():
    raise FileExistsError(f"Refusing to reuse a non-empty run directory: {WORK_ROOT}")
WORK_ROOT.mkdir(parents=True)
OUTPUT_ROOT.mkdir()

def run(command: list[str], *, cwd: Path | None = None) -> None:
    print("+", " ".join(command), flush=True)
    subprocess.run(command, cwd=cwd, check=True)

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

SOURCE_COMMIT

## 1. Xác minh private input

Notebook fail-closed nếu không tìm thấy đúng một manifest hoặc bất kỳ file nào sai SHA-256.

In [ ]:
job_candidates = sorted(INPUT_ROOT.rglob("kaggle_job.manifest.json"))
if len(job_candidates) != 1:
    raise RuntimeError(f"Expected exactly one kaggle_job.manifest.json, found {len(job_candidates)}")
JOB_MANIFEST = job_candidates[0]
BUNDLE_DIR = JOB_MANIFEST.parent
BUNDLE_MANIFEST = BUNDLE_DIR / "bundle.manifest.json"
if not BUNDLE_MANIFEST.is_file():
    raise FileNotFoundError(BUNDLE_MANIFEST)
bundle = json.loads(BUNDLE_MANIFEST.read_text(encoding="utf-8"))
if bundle.get("protocol") != "vifinqa_active_learning_kaggle_bundle_v1":
    raise ValueError("Unexpected bundle protocol")
for filename, record in bundle["files"].items():
    path = BUNDLE_DIR / filename
    if path.name != filename or not path.is_file():
        raise ValueError(f"Invalid bundle member: {filename}")
    if sha256_file(path) != record["sha256"] or path.stat().st_size != record["bytes"]:
        raise ValueError(f"Bundle hash/size mismatch: {filename}")
job = json.loads(JOB_MANIFEST.read_text(encoding="utf-8"))
assert job["status"] == "PREPARED_GPU_EXECUTION_NOT_RUN"
assert job["packet_count"] == 64
assert job["chatgpt_in_model_graph"] is False
assert job["training_eligible"] is False
assert job["certification_allowed"] is False
assert job["release_status"] == "blocked"
{"bundle_dir": str(BUNDLE_DIR), "packet_count": job["packet_count"], "hashes_verified": len(bundle["files"])}

## 2. Checkout source và CUDA preflight

Source được checkout ở commit bất biến. Runner tiếp tục tải từng model ở revision đã pin và kiểm tra SHA từng shard trước khi load 4-bit NF4.

In [ ]:
run(["git", "clone", "--filter=blob:none", REPO_URL, str(REPO_DIR)])
run(["git", "checkout", "--detach", SOURCE_COMMIT], cwd=REPO_DIR)
resolved_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
if resolved_commit != SOURCE_COMMIT:
    raise RuntimeError(f"Source commit mismatch: {resolved_commit}")
run([sys.executable, "-m", "pip", "install", "-q",
     "transformers==4.56.2", "accelerate==1.10.1", "bitsandbytes==0.47.0",
     "huggingface_hub==0.34.4", "sentencepiece==0.2.1", "safetensors==0.6.2"])
run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], cwd=REPO_DIR)
import torch
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable; enable a Kaggle GPU accelerator")
{"source_commit": resolved_commit, "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__}

## 3. Qwen3-8B proposer

Chạy 64 request với decoding xác định và Qwen thinking tắt. Raw output chưa có authority; bước kế tiếp mới kiểm tra closed schema và lineage.

In [ ]:
PROPOSER_DIR = OUTPUT_ROOT / "qwen3_8b_proposer"
run([sys.executable, "scripts/run_active_learning_open_source_model_v1.py",
     "--job-manifest", str(JOB_MANIFEST),
     "--role", "open_source_model_proposer",
     "--output-dir", str(PROPOSER_DIR)], cwd=REPO_DIR)
proposer_runtime = json.loads((PROPOSER_DIR / "raw_model_execution.manifest.json").read_text(encoding="utf-8"))
PROPOSER_VALIDATED = OUTPUT_ROOT / "qwen3_8b_proposer_validated_v1.jsonl"
run([sys.executable, "scripts/validate_active_learning_model_responses_v1.py",
     "--requests", str(BUNDLE_DIR / job["outputs"]["proposer_requests"]["path"]),
     "--raw-responses", proposer_runtime["outputs"]["raw_responses"]["path"],
     "--output", str(PROPOSER_VALIDATED)], cwd=REPO_DIR)
proposer_runtime["runtime_error_abstention_count"]

## 4. Mistral-Nemo-12B blind critic

Critic đọc request gốc trong private input, không đọc raw/validated output của proposer. Process riêng giúp giải phóng model Qwen trước khi load model 12B.

In [ ]:
CRITIC_DIR = OUTPUT_ROOT / "mistral_nemo_12b_critic"
run([sys.executable, "scripts/run_active_learning_open_source_model_v1.py",
     "--job-manifest", str(JOB_MANIFEST),
     "--role", "open_source_model_critic",
     "--output-dir", str(CRITIC_DIR)], cwd=REPO_DIR)
critic_runtime = json.loads((CRITIC_DIR / "raw_model_execution.manifest.json").read_text(encoding="utf-8"))
CRITIC_VALIDATED = OUTPUT_ROOT / "mistral_nemo_12b_critic_validated_v1.jsonl"
run([sys.executable, "scripts/validate_active_learning_model_responses_v1.py",
     "--requests", str(BUNDLE_DIR / job["outputs"]["critic_requests"]["path"]),
     "--raw-responses", critic_runtime["outputs"]["raw_responses"]["path"],
     "--output", str(CRITIC_VALIDATED)], cwd=REPO_DIR)
critic_runtime["runtime_error_abstention_count"]

## 5. Reconcile và fail-closed summary

Chỉ policy có cùng canonical hash mới được ghi là machine-provisional. Mọi output còn lại đi vào escalation. Cả hai nhánh đều không training/certification/submission eligible.

In [ ]:
RECONCILIATION_DIR = OUTPUT_ROOT / "reconciliation"
run([sys.executable, "scripts/reconcile_active_learning_model_responses_v1.py",
     "--packets", str(BUNDLE_DIR / job["outputs"]["packets"]["path"]),
     "--proposer-validated", str(PROPOSER_VALIDATED),
     "--critic-validated", str(CRITIC_VALIDATED),
     "--output-dir", str(RECONCILIATION_DIR)], cwd=REPO_DIR)
summary = json.loads((RECONCILIATION_DIR / "reconciliation_summary.json").read_text(encoding="utf-8"))
assert summary["packet_count"] == 64
assert summary["training_eligible_count"] == 0
assert summary["promotion_allowed_count"] == 0
assert summary["release_status"] == "blocked"
summary

## 6. Archive receipts

Download archive này để nhập lại repository và audit. Không dùng file machine-provisional làm training data trước authorized adjudication và independent probability audit.

In [ ]:
ARCHIVE_BASE = Path(f"/kaggle/working/vifinqa_active_learning_open_source_cycle_v1_{RUN_ID}")
ARCHIVE_PATH = ARCHIVE_BASE.with_suffix(".zip")
if ARCHIVE_PATH.exists():
    raise FileExistsError(ARCHIVE_PATH)
created = Path(shutil.make_archive(str(ARCHIVE_BASE), "zip", root_dir=OUTPUT_ROOT))
{"archive": str(created), "sha256": sha256_file(created), "bytes": created.stat().st_size, "release_status": "blocked"}